In [ ]:
! pip install scikit-allel
import numpy as np
import tqdm
import allel
import sklearn.metrics
import matplotlib.pyplot as plt
import glob

In [ ]:
# Downloading VCFs
remote_dir="gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/v3/15x/workpackage_1_ultralong_annotated/merged/scored"
svtype = "del"
! rm -rf ./scored-vcfs/
! mkdir ./scored-vcfs/
! gsutil -m cp "{remote_dir}/del_*_score.vcf.gz*" ./scored-vcfs/

# Q100 STVAR BED
! gsutil -m cp "gs://fc-7f861a33-ddb4-4b2f-8d10-5679c9df6108/v3/ultralong_inputs/GRCh38_HG2-T2TQ100-V1.1_stvar.benchmark.bed" ./confident.bed
#REGIONS_FILE_STRING="--regions-file confident.bed --regions-overlap record"  # Same overlap criterion as training script
REGIONS_FILE_STRING=" "


! rm -rf ./filtered/
! mkdir ./filtered/
MIN=10_000    # 10_000 25_000 50_000 100_000 2_000_000
MAX=2_000_000 
! bcftools filter {REGIONS_FILE_STRING} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z scored-vcfs/{svtype}_custom_score.vcf.gz --output filtered/{svtype}_custom_score.vcf.gz
! tabix filtered/{svtype}_custom_score.vcf.gz
! bcftools filter {REGIONS_FILE_STRING} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z scored-vcfs/{svtype}_fex_score.vcf.gz --output filtered/{svtype}_fex_score.vcf.gz
! tabix filtered/{svtype}_fex_score.vcf.gz
! bcftools filter {REGIONS_FILE_STRING} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z scored-vcfs/{svtype}_all_score.vcf.gz --output filtered/{svtype}_all_score.vcf.gz
! tabix filtered/{svtype}_all_score.vcf.gz


scored_vcfs = [f"filtered/{svtype}_custom_score.vcf.gz",
               f"filtered/{svtype}_fex_score.vcf.gz",
               f"filtered/{svtype}_all_score.vcf.gz"
]
vcf_names = ['custom',
             'feature_extraction.py',
             'all'
]

In [12]:
# Plotting logic
def load_callsets(scored_vcfs):
    callsets = []
    for i, scored_vcf in tqdm.tqdm(enumerate(scored_vcfs)):
        callset = allel.read_vcf(scored_vcf, fields=['variants/SVTYPE',
                                                     'variants/SVLEN',
                                                     'variants/SCORE',
                                                     'variants/CALIBRATION_SENSITIVITY',
                                                     'variants/training',
                                                     'variants/extracted',
                                                     'variants/CHROM',
                                                     'variants/SUPP_PBSV',
                                                     'variants/SUPP_SNIFFLES',
                                                     'variants/SUPP_PAV'
                                                    ])
        callsets.append(callset)
    return callsets

# def plot_roc(callsets, calibration_sensitivity_thresholds=[0.7,0.9]):
#     # Masks
#     # Remark: we trained on chr6..Y. We test here on chr1..5.
#     test_mask_lambda = lambda callset: ((callset['variants/CHROM'] == 'chr1') | 
#                                         (callset['variants/CHROM'] == 'chr2') | 
#                                         (callset['variants/CHROM'] == 'chr3') | 
#                                         (callset['variants/CHROM'] == 'chr4') | 
#                                         (callset['variants/CHROM'] == 'chr5'))
    
#     fig = plt.figure(figsize=(5, 5))
#     for mask_title, mask_color, additional_mask_lambda in [
#                                                ['all', 'blue',
#                                                 lambda callset: test_mask_lambda(callset) ]]:
        
#         for i, callset in tqdm.tqdm(enumerate(callsets)):
#             additional_mask = additional_mask_lambda(callset)
            
#             # Score ROC
#             metrics_mask = ~np.isnan(callset['variants/SCORE']) & additional_mask
#             fpr, tpr, thresholds = sklearn.metrics.roc_curve(
#                 y_true=callset['variants/training'][metrics_mask], 
#                 y_score=callset['variants/SCORE'][metrics_mask])
#             plt.plot(fpr, tpr, label=mask_title if i==0 else None, c=mask_color, alpha=0.25)
            
#             # CAL_SENS thresholds
#             metrics_mask = ~np.isnan(callset['variants/CALIBRATION_SENSITIVITY']) & additional_mask
#             fpr, tpr, thresholds = sklearn.metrics.roc_curve(
#                 y_true=callset['variants/training'][metrics_mask], 
#                 y_score=1 - callset['variants/CALIBRATION_SENSITIVITY'][metrics_mask])
#             fpr_selection = fpr[[np.argmax(tpr >= cst) for cst in calibration_sensitivity_thresholds]]
#             tpr_selection = tpr[[np.argmax(tpr >= cst) for cst in calibration_sensitivity_thresholds]]
#             plt.scatter(fpr_selection, tpr_selection, label=None, 
#                         c=mask_color, marker='d', s=4)

#     plt.xlabel('False Positive Rate')
#     plt.ylabel('True Positive Rate')
#     plt.title(f'Deletions >=10kb, chr1..5, 145 HPRC+HGSVC samples, GRCh38.')
#     plt.legend()
#     plt.show()
    

    
import matplotlib.pyplot as plt
import matplotlib.cm as cm # For color maps
import numpy as np
import sklearn.metrics
import tqdm
import os

def plot_roc(callsets, vcf_names=None):
    """
    Plots individual ROC curves for each VCF in callsets.
    :param callsets: List of dictionaries loaded via allel.read_vcf
    :param vcf_names: Optional list of strings for legend names
    """
    # Define a mask for the test chromosomes
    test_mask_lambda = lambda callset: np.isin(callset['variants/CHROM'],
                                               ['chr1', 'chr2', 'chr3'])
# PRODUCTION:                                  ['chr1', 'chr2', 'chr3'])
# FOR INV:                                     ['chr1', 'chr4', 'chr7', 'chr10', 'chr13', 'chr16', 'chr19', 'chr22'])
# ORIGINAL:                                    ['chr1', 'chr2', 'chr3', 'chr4', 'chr5'])
# ALL:                                         [ 'chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22'])
    fig, ax = plt.subplots(figsize=(7, 7))
    
    # Setup colors: generate a list of distinct colors based on the number of VCFs
    num_vcf = len(callsets)
    colors = cm.get_cmap('tab10', num_vcf).colors if num_vcf <= 10 else cm.rainbow(np.linspace(0, 1, num_vcf))

    for i, callset in tqdm.tqdm(enumerate(callsets), total=num_vcf):
        # 1. Setup metadata for this specific VCF
        vcf_label = vcf_names[i] if vcf_names else f"VCF {i+1}"
        print(f"Processing Index {i}: Labeling curve as '{vcf_label}' File name: '{scored_vcfs[i]}'")
        current_color = colors[i]
        additional_mask = test_mask_lambda(callset)
        
        # 2. Calculate and Plot Score ROC
        # We ensure SCORE is not NaN and within our target chromosomes
        metrics_mask = ~np.isnan(callset['variants/SCORE']) & additional_mask
        
        if np.any(metrics_mask):
            fpr, tpr, _ = sklearn.metrics.roc_curve(
                y_true=callset['variants/training'][metrics_mask], 
                y_score=callset['variants/SCORE'][metrics_mask]
            )
            # Plot the line with its unique label and color
            roc_auc = sklearn.metrics.auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f"{vcf_label} (AUC={roc_auc:.3f})", color=current_color, lw=2, alpha=0.8)
            
            
    # GT_COUNT points
    callset = callsets[0]
    additional_mask = test_mask_lambda(callset)
    y_true_all = callset['variants/training'][additional_mask]
        
    gt_mask = (callset['variants/SUPP_PBSV'] > 0) & additional_mask
    y_pred_gt = gt_mask[additional_mask].astype(int) 
    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true_all, y_pred_gt).ravel()
    point_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    point_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    ax.scatter(point_fpr, point_tpr, color='white', marker='s', s=100, edgecolors='black', zorder=6, label=f"SUPP pbsv")
    
    gt_mask = (callset['variants/SUPP_SNIFFLES'] > 0) & additional_mask
    y_pred_gt = gt_mask[additional_mask].astype(int) 
    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true_all, y_pred_gt).ravel()
    point_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    point_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    ax.scatter(point_fpr, point_tpr, color='white', marker='^', s=100, edgecolors='black', zorder=6, label=f"SUPP Sniffles")
    
    gt_mask = (callset['variants/SUPP_PAV'] > 0) & additional_mask
    y_pred_gt = gt_mask[additional_mask].astype(int) 
    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true_all, y_pred_gt).ravel()
    point_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    point_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    ax.scatter(point_fpr, point_tpr, color='white', marker='o', s=100, edgecolors='black', zorder=6, label=f"SUPP PAV")
    

    # Aesthetics
    ax.plot([0, 1], [0, 1], color='grey', linestyle='--', alpha=0.5) # Diagonal baseline
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
# FOR INV:     ax.set_title(f"{svtype} in ({MIN}..{MAX}], chr{1,4,7,10,13,16,19,22}, 292 HPRC and HGSVC samples, GRCh38")
# PRODUCTION:  ax.set_title(f"{svtype} in ({MIN}..{MAX}], chr1..3, 292 HPRC and HGSVC samples, GRCh38")
    ax.set_title(f"{svtype} in ({MIN}..{MAX}], chr1..3, 292 HPRC and HGSVC samples, GRCh38")
    ax.legend(loc='lower right', fontsize='small', frameon=True)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [ ]:
callsets = load_callsets(scored_vcfs)
plot_roc(callsets, vcf_names=vcf_names)